In [28]:
!pip install transformers peft datasets

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

data = load_dataset("glue", "sst2")
train_val = data['train']
train_data = train_val.select(range(0, len(train_val)-1000))
val_data   = train_val.select(range(len(train_val)-1000, len(train_val)))
test_data  = data['validation']

model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

def preprocess_function(examples):
    return tokenizer(examples["sentence"], truncation=True, padding="max_length", max_length=128)
train_dataset = train_data.map(preprocess_function, batched=True)
val_dataset   = val_data.map(preprocess_function, batched=True)
test_dataset  = test_data.map(preprocess_function, batched=True)


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [26]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'true'

In [84]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch, tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tok   = AutoTokenizer.from_pretrained("roberta-base")
model = AutoModelForMaskedLM.from_pretrained("roberta-base").to(device).eval()

In [128]:
base = model

In [126]:
TEMPLATE   = "[CLS] Review: {text}. Sentiment of review (only positive or negative): <mask>."
id_pos = [tok.convert_tokens_to_ids(t) for t in ["Ġgood", "Ġexcellent", "Ġpositive"]]
id_neg = [tok.convert_tokens_to_ids(t) for t in ["Ġbad",  "Ġterrible",  "Ġnegative"]]

def predict_label(mask_logits):
    pos_score = mask_logits[id_pos].max()
    neg_score = mask_logits[id_neg].max()
    return 1 if pos_score > neg_score else 0
    
sst2 = load_dataset("glue", "sst2", split="validation")
sentences = sst2["sentence"]
labels    = sst2["label"]
print(id_pos, id_neg)
batch_size = 64
correct = total = 0

for start in tqdm.tqdm(range(0, len(sentences), batch_size)):
    batch_sents = sentences[start:start+batch_size]
    batch_labels = labels[start:start+batch_size]

    prompts = [TEMPLATE.format(text=s) for s in batch_sents]
    inputs  = tok(prompts, return_tensors="pt", padding=True,
                  truncation=True, max_length=256).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits

    mask_pos = (inputs.input_ids == tok.mask_token_id).nonzero(as_tuple=False)

    for row_idx, col_idx in mask_pos:
        m_logits = logits[row_idx, col_idx]
        top_id   = m_logits.argmax().item()
        top_word = tok.decode(top_id).strip()
        pred     = predict_label(m_logits)
        gold     = batch_labels[row_idx]
        correct += int(pred == gold)
        total   += 1

print(f"Dev accuracy with hard prompt = {correct/total:.3%}")

[205, 4206, 1313] [1099, 6587, 2430]


100%|██████████| 14/14 [00:02<00:00,  5.09it/s]

Dev accuracy with hard prompt = 85.092%


In [165]:
import torch, time, itertools, json
from datasets import load_dataset
from transformers import (AutoTokenizer,
                          AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
from peft import PrefixTuningConfig, TaskType, get_peft_model
from evaluate import load as load_metric

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
metric = load_metric("accuracy")
tok    = AutoTokenizer.from_pretrained("roberta-base")

raw      = load_dataset("glue", "sst2")
train_ds = raw["train"]
val_ds   = raw["validation"]

def tok_f(batch):
    return tok(batch["sentence"], truncation=True,
               padding="max_length", max_length=128)

def prepare(ds):
    ds = ds.map(tok_f, batched=True, remove_columns=["sentence", "idx"])
    ds = ds.rename_column("label", "labels")
    ds.set_format(type="torch",
                  columns=["input_ids", "attention_mask", "labels"])
    return ds

train_ds = prepare(train_ds)
val_ds   = prepare(val_ds)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    return metric.compute(predictions=preds, references=labels)

lrs        = [5e-4, 1e-3, 2e-3]
vtok_lens  = [10, 20, 30]

results = []

for lr, vlen in itertools.product(lrs, vtok_lens):
    print(f"\n=== lr={lr} | virtual_tokens={vlen} ===")

    base = AutoModelForSequenceClassification.from_pretrained(
        "roberta-base", num_labels=2).to(device)
    for p in base.roberta.parameters():
        p.requires_grad = False

    peft_cfg = PrefixTuningConfig(
        task_type          = TaskType.SEQ_CLS,
        num_virtual_tokens = vlen,
        prefix_projection  = False
    )
    model = get_peft_model(base, peft_cfg).to(device)

    args = TrainingArguments(
        output_dir          = f"./ptv2_lr{lr}_v{vlen}",
        eval_strategy = "epoch",
        num_train_epochs    = 1,
        per_device_train_batch_size = 16,
        per_device_eval_batch_size  = 16,
        learning_rate       = lr,
        fp16                = torch.cuda.is_available(),
        save_strategy       = "no",
        report_to           = "none",
        seed                = 42,
    )

    trainer = Trainer(model=model, args=args,
                      train_dataset=train_ds,
                      eval_dataset=val_ds,
                      compute_metrics=compute_metrics)

    t0 = time.time()
    trainer.train()
    dev_acc = trainer.evaluate()["eval_accuracy"]
    elapsed = int(time.time() - t0)

    res = {"lr": lr, "virtual_tokens": vlen,
           "dev_accuracy": round(dev_acc, 4),
           "sec": elapsed}
    print(json.dumps(res))
    results.append(res)

    del model, trainer, base
    torch.cuda.empty_cache()

print("\n=== grid search summary (sorted) ===")
for r in sorted(results, key=lambda x: -x["dev_accuracy"]):
    print(json.dumps(r))


Map:   0%|          | 0/872 [00:00<?, ? examples/s]


=== lr=0.0005 | virtual_tokens=10 ===


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.521800,0.423190,0.818807


{"lr": 0.0005, "virtual_tokens": 10, "dev_accuracy": 0.8188, "sec": 575}

=== lr=0.0005 | virtual_tokens=20 ===


...

Epoch,Training Loss,Validation Loss,Accuracy
1,0.297800,0.247648,0.900229


{"lr": 0.002, "virtual_tokens": 20, "dev_accuracy": 0.9002, "sec": 577}

=== lr=0.002 | virtual_tokens=30 ===


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.270800,0.235707,0.908257


{"lr": 0.002, "virtual_tokens": 30, "dev_accuracy": 0.9083, "sec": 581}

=== grid search summary (sorted) ===
{"lr": 0.002, "virtual_tokens": 30, "dev_accuracy": 0.9083, "sec": 581}
{"lr": 0.002, "virtual_tokens": 20, "dev_accuracy": 0.9002, "sec": 577}
{"lr": 0.002, "virtual_tokens": 10, "dev_accuracy": 0.8991, "sec": 576}
{"lr": 0.001, "virtual_tokens": 30, "dev_accuracy": 0.8968, "sec": 580}
{"lr": 0.0005, "virtual_tokens": 30, "dev_accuracy": 0.883, "sec": 581}
{"lr": 0.001, "virtual_tokens": 20, "dev_accuracy": 0.8807, "sec": 575}
{"lr": 0.001, "virtual_tokens": 10, "dev_accuracy": 0.867, "sec": 575}
{"lr": 0.0005, "virtual_tokens": 20, "dev_accuracy": 0.8521, "sec": 577}
{"lr": 0.0005, "virtual_tokens": 10, "dev_accuracy": 0.8188, "sec": 575}


In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
import torch, time, itertools, json
from datasets import load_dataset
from transformers import (AutoTokenizer,
                          AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
from peft import PrefixTuningConfig, TaskType, get_peft_model
from evaluate import load as load_metric

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
metric = load_metric("accuracy")
tok    = AutoTokenizer.from_pretrained("roberta-base")

raw      = load_dataset("glue", "sst2")
train_ds = raw["train"]
val_ds   = raw["validation"]

def tok_f(batch):
    return tok(batch["sentence"], truncation=True,
               padding="max_length", max_length=128)

def prepare(ds):
    ds = ds.map(tok_f, batched=True, remove_columns=["sentence", "idx"])
    ds = ds.rename_column("label", "labels")
    ds.set_format(type="torch",
                  columns=["input_ids", "attention_mask", "labels"])
    return ds

train_ds = prepare(train_ds)
val_ds   = prepare(val_ds)
base = AutoModelForSequenceClassification.from_pretrained(
        "roberta-base", num_labels=2).to(device)

for p in base.roberta.parameters():
    p.requires_grad = False

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    return metric.compute(predictions=preds, references=labels)

peft_cfg = PrefixTuningConfig(
    task_type          = TaskType.SEQ_CLS,
    num_virtual_tokens = 30,
    prefix_projection  = False
)
model = get_peft_model(base, peft_cfg).to(device)

args = TrainingArguments(
    output_dir          = "./ptuningv2",
    eval_strategy = "epoch",
    save_strategy       = "epoch",
    load_best_model_at_end = True,
    metric_for_best_model  = "eval_accuracy",
    greater_is_better      = True,

    num_train_epochs    = 3,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,
    learning_rate       = 2e-3,
    fp16                = torch.cuda.is_available(),
    save_total_limit    = 2,
    report_to           = "none",
    seed                = 42,
)

trainer = Trainer(model=model,
                  args=args,
                  train_dataset=train_ds,
                  eval_dataset=val_ds,
                  compute_metrics=compute_metrics,
)

trainer.train()
print("Лучший dev-accuracy:", trainer.evaluate()["eval_accuracy"])

trainer.save_model("./roberta_ptv2_best")
tokenizer.save_pretrained("./roberta_ptv2_best")

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Epoch,Training Loss,Validation Loss


In [3]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 8.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.12.0 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.8.4.1 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cudnn-cu12==9.1.0.70; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cudnn-cu12 9.3.0.75 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cufft-cu12==1

In [145]:
from evaluate import load
acc_metric = load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    
    return acc_metric.compute(predictions=preds, references=labels)

trainer2 = Trainer(
    model           = model,
    args            = args,
    train_dataset   = train_ds,
    eval_dataset    = val_ds,
    compute_metrics = compute_metrics
)

print("Dev accuracy :", trainer2.evaluate(val_ds))

print("Test accuracy:", trainer2.evaluate(test_ds))


No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Dev accuracy : {'eval_loss': 0.24817104637622833, 'eval_model_preparation_time': 0.003, 'eval_accuracy': 0.906, 'eval_runtime': 3.9427, 'eval_samples_per_second': 253.631, 'eval_steps_per_second': 15.979}
Test accuracy: {'eval_loss': 0.23937082290649414, 'eval_model_preparation_time': 0.003, 'eval_accuracy': 0.908256880733945, 'eval_runtime': 3.4798, 'eval_samples_per_second': 250.591, 'eval_steps_per_second': 15.806}


In [155]:
test = prepare(data["test"])

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

In [160]:
preds_sst2 = []
batch_size = 64
for i in tqdm.tqdm(range(0, len(test), batch_size)):
    batch = test[i : i + batch_size]
    with torch.no_grad():
        out = model(
            input_ids      = batch["input_ids"].to(device),
            attention_mask = batch["attention_mask"].to(device)
        ).logits

    preds_sst2.extend(out.argmax(-1).cpu().tolist())  

100%|██████████| 29/29 [00:06<00:00,  4.36it/s]
